### Họ và Tên: Trần Nhật Trường
### MSSV: 24120486

# Bài tập về nhà buổi 7

## Bài 1: Titanic Dataset

### chuẩn bị môi trường

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler, LabelEncoder 
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score ,recall_score, precision_score,f1_score, confusion_matrix, log_loss
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)      
print("Sẵn sàng.")

Sẵn sàng.


### Load data

In [3]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### Xử lí missing values

In [4]:
print("Tỷ lệ giá trị thiếu (missing values) của từng cột ban đầu:")
print(df.isnull().mean())

leaky = ['alive', 'who', 'adult_male', 'class', 'deck', 'embark_town', 'alone']      # điền danh sách cột cần bỏ (chỉ những cột có trong df)
df = df.drop(columns=leaky)

print("\nCác cột còn lại sau khi lọc bỏ cột rò rỉ/dư thừa:")
print(list(df.columns))


Tỷ lệ giá trị thiếu (missing values) của từng cột ban đầu:
survived       0.000000
pclass         0.000000
sex            0.000000
age            0.198653
sibsp          0.000000
parch          0.000000
fare           0.000000
embarked       0.002245
class          0.000000
who            0.000000
adult_male     0.000000
deck           0.772166
embark_town    0.002245
alive          0.000000
alone          0.000000
dtype: float64

Các cột còn lại sau khi lọc bỏ cột rò rỉ/dư thừa:
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


### Chia dữ liệu

In [5]:
X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

print("Kích thước tập Train và Test sau khi chia:")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

survivedRate_train = round(len(y_train[y_train==1])/len(y_train),2)
survivedRate_test = round(len(y_test[y_test==1])/len(y_test),2)
print(f"Tỷ lệ hành khách sống sót (survived = 1) trên tập Train: {survivedRate_train}")
print(f"Tỷ lệ hành khách sống sót (survived = 1) trên tập Test:  {survivedRate_test}")


Kích thước tập Train và Test sau khi chia:
X_train: (712, 7), X_test: (179, 7)
Tỷ lệ hành khách sống sót (survived = 1) trên tập Train: 0.38
Tỷ lệ hành khách sống sót (survived = 1) trên tập Test:  0.39


### Pipeline xử lí dữ liệu

In [6]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy='median')),
    ("scaler", StandardScaler()), # Chuẩn hóa biến số thực tại đây để tránh scale biến category
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("onehot",  OneHotEncoder()),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)               # fit CHỈ trên train
X_train_scaled = preprocess.transform(X_train)
X_test_scaled = preprocess.transform(X_test)

print("--- Tiền xử lý dữ liệu ---")
print("Kích thước tập Train sau tiền xử lý:", X_train_scaled.shape)
print("Kích thước tập Test sau tiền xử lý:", X_test_scaled.shape)
print("Danh sách đặc trưng đầu ra của ColumnTransformer:")
print(list(preprocess.get_feature_names_out()))

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


--- Tiền xử lý dữ liệu ---
Kích thước tập Train sau tiền xử lý: (712, 10)
Kích thước tập Test sau tiền xử lý: (179, 10)
Danh sách đặc trưng đầu ra của ColumnTransformer:
['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_female', 'cat__sex_male', 'cat__embarked_C', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass']


### Tiến hành xây dựng model logistic regression

In [7]:
lg_model= LogisticRegression()
lg_model.fit(X_train_scaled,y_train)

y_pred_lg = lg_model.predict(X_test_scaled)
Acc_lg = accuracy_score(y_test,y_pred_lg)
Precision_lg = precision_score(y_test,y_pred_lg)
recall_lg = recall_score(y_test,y_pred_lg)
f1_lg= f1_score(y_test,y_pred_lg)
conf_matrix_lg = confusion_matrix(y_test,y_pred_lg)

print("Mô hình Logistic Regression:")
print(f"Accuracy:  {Acc_lg:.4f}")
print(f"Precision: {Precision_lg:.4f}")
print(f"Recall:    {recall_lg:.4f}")
print(f"F1-score:  {f1_lg:.4f}")
print("Confusion Matrix:")
print(conf_matrix_lg)


Mô hình Logistic Regression:
Accuracy:  0.8045
Precision: 0.7931
Recall:    0.6667
F1-score:  0.7244
Confusion Matrix:
[[98 12]
 [23 46]]


### Vì trong bài tập về nhà tuần trước thì chỉ dùng dataset titanic để làm sạch còn dùng california_housing để xây dựng Linear Regression nên em sẽ xây dựng lại để so sánh

In [8]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

y_pred_lr_binary = (y_pred_lr >= 0.5).astype(int)

Acc_lr = accuracy_score(y_test, y_pred_lr_binary)
Precision_lr = precision_score(y_test, y_pred_lr_binary)
recall_lr = recall_score(y_test, y_pred_lr_binary)
f1_lr = f1_score(y_test, y_pred_lr_binary)
conf_matrix_lr = confusion_matrix(y_test, y_pred_lr_binary)

print("Mô hình Linear Regression (phân loại bằng cách áp ngưỡng 0.5):")
print(f"Accuracy:  {Acc_lr:.4f}")
print(f"Precision: {Precision_lr:.4f}")
print(f"Recall:    {recall_lr:.4f}")
print(f"F1-score:  {f1_lr:.4f}")
print("Confusion Matrix:")
print(conf_matrix_lr)


Mô hình Linear Regression (phân loại bằng cách áp ngưỡng 0.5):
Accuracy:  0.8101
Precision: 0.7869
Recall:    0.6957
F1-score:  0.7385
Confusion Matrix:
[[97 13]
 [21 48]]


### Nhận xét so sánh hiệu năng giữa Logistic Regression và Linear Regression trên Titanic Dataset:

1. Kết quả các chỉ số hiệu năng (trên tập Test):
   * Logistic Regression: Accuracy ~ 80.45%, Precision ~ 79.31%, Recall ~ 66.67%, F1-score ~ 72.44%.
   * Linear Regression (áp ngưỡng 0.5): Accuracy ~ 81.01%, Precision ~ 78.69%, Recall ~ 69.57%, F1-score ~ 73.85%.

2. Tại sao Logistic Regression vẫn là mô hình phù hợp hơn đối với bài toán phân loại sống sót?
   * Vì có đầu ra xác suất hợp lệ, cho biết xác xuất sống sót của 1 hành khách thay vì chỉ dự đoán sống chết như linear regression.
   * Ít nhạy với giá trị ngoại lai hơn so với linear regression, linear regression dùng bình phương tối tiểu để tối ưu nên đường ranh giới có thể bị lệch bởi ngoại lai còn logistic dùng hàm phi tuyến để dự đoán trực tiếp cho từng giá trị nên ít bị ảnh hưởng hơn.
   * Linear Regression tối ưu hóa Mean Squared Error (MSE) cho biến liên tục, trong khi Logistic Regression tối ưu hóa Log-loss trực tiếp cho phân loại xác suất nhị phân, giúp mô hình ổn định và tổng quát hóa tốt hơn khi dữ liệu thực tế thay đổi.

## Bài 2: Dry Bean Dataset

### Load data

In [9]:
df_train = pd.read_csv('Dry_Bean_Dataset/dry_bean_train.csv')
df_test = pd.read_csv('Dry_Bean_Dataset/dry_bean_test.csv')
print("Kích thước tập dữ liệu Dry Bean Train:", df_train.shape)
print("Kích thước tập dữ liệu Dry Bean Test:", df_test.shape)

print("5 dòng đầu của tập Train:")
display(df_train.head())

print("5 dòng đầu của tập Test:")
display(df_test.head())


Kích thước tập dữ liệu Dry Bean Train: (10834, 17)
Kích thước tập dữ liệu Dry Bean Test: (2709, 17)
5 dòng đầu của tập Train:


,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,69471,1069.638,399.100245,225.005782,1.773733,0.825923,71088,297.410868,0.707386,0.977254,0.763027,0.745203,0.005745,0.001093,0.555328,0.985004,CALI
1,82877,1162.581,391.817013,270.836144,1.446694,0.722634,84171,324.841921,0.825986,0.984627,0.770544,0.829065,0.004728,0.001378,0.687349,0.994384,BARBUNYA
2,65042,1023.506,419.202858,198.962774,2.106941,0.880190,65748,287.774298,0.783403,0.989262,0.780231,0.686480,0.006445,0.000883,0.471255,0.992906,HOROZ
3,41315,758.920,287.438268,183.447580,1.566869,0.769858,41704,229.355383,0.791930,0.990672,0.901417,0.797929,0.006957,0.001740,0.636691,0.997611,SIRA
4,91088,1168.645,459.300729,253.950486,1.808623,0.833243,91799,340.553731,0.789051,0.992255,0.838119,0.741461,0.005042,0.000940,0.549765,0.994318,CALI


5 dòng đầu của tập Test:


,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,40512,737.636,248.202030,208.525152,1.190274,0.542365,40973,227.115566,0.744268,0.988749,0.935641,0.915043,0.006127,0.002650,0.837304,0.996621,SEKER
1,31890,660.655,250.417300,162.522502,1.540816,0.760783,32240,201.503372,0.801962,0.989144,0.918153,0.804670,0.007853,0.002031,0.647494,0.997670,DERMASON
2,33107,671.869,244.728317,172.938116,1.415121,0.707560,33506,205.312303,0.798683,0.988092,0.921638,0.838940,0.007392,0.002259,0.703820,0.995990,DERMASON
3,61684,984.878,412.115648,191.473280,2.152340,0.885515,62303,280.247227,0.784724,0.990065,0.799130,0.680021,0.006681,0.000881,0.462428,0.995303,HOROZ
4,37189,700.253,243.343401,194.910062,1.248491,0.598708,37534,217.601713,0.784826,0.990808,0.953047,0.894217,0.006543,0.002581,0.799623,0.998322,SEKER


In [10]:
y_train = df_train['class']
X_train = df_train.drop('class', axis = 1)
y_test = df_test['class']
X_test = df_test.drop('class', axis = 1)

print("Tách nhãn và thuộc tính cho bộ dữ liệu Dry Bean:")
print("Kích thước nhãn y_train:", y_train.shape)
print("Kích thước thuộc tính X_train:", X_train.shape)
print("Kích thước nhãn y_test:", y_test.shape)
print("Kích thước thuộc tính X_test:", X_test.shape)


Tách nhãn và thuộc tính cho bộ dữ liệu Dry Bean:
Kích thước nhãn y_train: (10834,)
Kích thước thuộc tính X_train: (10834, 16)
Kích thước nhãn y_test: (2709,)
Kích thước thuộc tính X_test: (2709, 16)


In [11]:
encoder = LabelEncoder()
y_train_encoder = encoder.fit_transform(y_train)
y_test_encoder = encoder.transform(y_test)

print("Mã hóa nhãn lớp (Label Encoding):")
print("5 nhãn đầu tiên của y_train sau mã hóa:", y_train_encoder[:5])
print("5 nhãn đầu tiên của y_test sau mã hóa:", y_test_encoder[:5])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nChuẩn hóa dữ liệu đầu vào (Standard Scaling):")
print("Kích thước X_train sau khi chuẩn hóa:", X_train_scaled.shape)
print("Kích thước X_test sau khi chuẩn hóa:", X_test_scaled.shape)


Mã hóa nhãn lớp (Label Encoding):
5 nhãn đầu tiên của y_train sau mã hóa: [2 0 4 6 2]
5 nhãn đầu tiên của y_test sau mã hóa: [5 3 3 4 5]

Chuẩn hóa dữ liệu đầu vào (Standard Scaling):
Kích thước X_train sau khi chuẩn hóa: (10834, 16)
Kích thước X_test sau khi chuẩn hóa: (2709, 16)


### Thuật toán Logistic Regression

In [12]:
from sklearn.metrics import classification_report

model_lg2 = LogisticRegression(max_iter=1000) # tăng max_iter để tránh cảnh báo hội tụ
model_lg2.fit(X_train_scaled, y_train_encoder)
y_pred_lg2 = model_lg2.predict(X_test_scaled)
y_prob = model_lg2.predict_proba(X_test_scaled)

Acc_lg2 = accuracy_score(y_test_encoder, y_pred_lg2)
loss_value = log_loss(y_test_encoder, y_prob)

print("Mô hình Logistic Regression cho Dry Bean Dataset:")
print(f"Accuracy: {Acc_lg2:.4f}")
print(f"Log Loss: {loss_value:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_encoder, y_pred_lg2, target_names=encoder.classes_))

conf_matrix_lg2 = confusion_matrix(y_test_encoder, y_pred_lg2)
print("Confusion Matrix:")
print(conf_matrix_lg2)


Mô hình Logistic Regression cho Dry Bean Dataset:
Accuracy: 0.9192
Log Loss: 0.2183

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709

Confusion Matrix:
[[236   0  18   0   0   4   7]
 [  0 104   0   0   0   0   0]
 [  8   0 307   0   5   2   4]
 [  0   0   0 643   0  13  53]
 [  1   0  11   5 351   0   4]
 [  9   0   0   5   0 383   9]
 [  1   0   1  41   8  10 466]]


### KNN

In [14]:
from sklearn.model_selection import GridSearchCV

knn_base = KNeighborsClassifier()
param_grid = {'n_neighbors': list(range(3, 30, 2))}

grid_search = GridSearchCV(knn_base, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train_encoder)

k_best = grid_search.best_params_['n_neighbors']
best_cv_score = grid_search.best_score_
print(f"k tối ưu tìm được qua Cross-Validation là: {k_best} với score trung bình trên tập Train: {best_cv_score:.4f}")

knn_best = grid_search.best_estimator_
y_pred_knn = knn_best.predict(X_test_scaled)
Acc_knn = accuracy_score(y_test_encoder, y_pred_knn)

print(f"\nĐánh giá KNN (k={k_best}) trên tập Test thực tế:")
print(f"Accuracy: {Acc_knn:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_encoder, y_pred_knn, target_names=encoder.classes_))

conf_matrix_knn = confusion_matrix(y_test_encoder, y_pred_knn)
print("Confusion Matrix:")
print(conf_matrix_knn)


k tối ưu tìm được qua Cross-Validation là: 13 với score trung bình trên tập Train: 0.9259

Đánh giá KNN (k=13) trên tập Test thực tế:
Accuracy: 0.9162

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.95      0.88      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.90      0.96      0.93       326
    DERMASON       0.91      0.91      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.94      0.94      0.94       406
        SIRA       0.85      0.87      0.86       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709

Confusion Matrix:
[[233   0  20   0   1   4   7]
 [  0 104   0   0   0   0   0]
 [  3   0 312   0   6   2   3]
 [  0   0   0 644   0  13  52]
 [  0   0  14   5 345   0   8]
 [  5   0   0   7   0 383  11]
 [  4   0   1  50   6   

#### kết quả hiệu năng (trên tập Test):
   -  Logistic Regression: Accuracy ~ 91.92%, Log Loss ~ 0.2183.
   - KNN (với k=13 tối ưu tìm từ Cross-Validation): Accuracy ~ 91.62%.
   Cả 2 đều cho kết quả tốt với data này.